# Homework 5: CTE & Materialized View

## 1. Setup and loading

In [1]:
#Setup the environment
%pip install -q pgserver psycopg2-binary sqlalchemy pandas

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [2]:
# Load the Olist CSV files, downloading the archive when any file is missing
from pathlib import Path
import hashlib
from shutil import copyfileobj
from tempfile import TemporaryFile
from urllib.request import urlopen
from zipfile import ZipFile

DATASET_URL = "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce"
EXPECTED_SHA256 = {
    'olist_customers_dataset.csv': '983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2',
    'olist_orders_dataset.csv': '8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf',
    'olist_order_items_dataset.csv': '0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279',
    'olist_products_dataset.csv': '3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc',
    'olist_sellers_dataset.csv': '1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e',
}

files = {
    'olist_customers_raw': 'data/olist/olist_customers_dataset.csv',
    'olist_orders_raw': 'data/olist/olist_orders_dataset.csv',
    'olist_order_items_raw': 'data/olist/olist_order_items_dataset.csv',
    'olist_products_raw': 'data/olist/olist_products_dataset.csv',
    'olist_sellers_raw': 'data/olist/olist_sellers_dataset.csv',
}

file_paths = {}

for dataset_name, csv_file_name in files.items():
    file_paths[dataset_name] = Path(csv_file_name)

missing_files = [path for path in file_paths.values() if not path.is_file()]
if missing_files:
    with urlopen(DATASET_URL) as response, TemporaryFile() as archive_file:
        copyfileobj(response, archive_file)
        archive_file.seek(0)
        with ZipFile(archive_file) as archive:
            members = {Path(name).name: name for name in archive.namelist()}
            absent = [path.name for path in missing_files if path.name not in members]
            if absent:
                raise FileNotFoundError(f'Missing CSV files in dataset archive: {absent}')
            for path in missing_files:
                path.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(members[path.name]) as source, path.open('wb') as target:
                    copyfileobj(source, target)

# Check both checked-in and freshly downloaded CSVs before loading raw tables.
for file_path in file_paths.values():
    digest = hashlib.sha256()
    with file_path.open('rb') as source_file:
        for chunk in iter(lambda: source_file.read(1024 * 1024), b''):
            digest.update(chunk)
    actual = digest.hexdigest()
    expected = EXPECTED_SHA256[file_path.name]
    if actual != expected:
        raise ValueError(
            f'SHA-256 mismatch for {file_path}: expected {expected}, got {actual}. '
            'Use the dataset snapshot documented in README.md.'
        )

for table_name, file_path in file_paths.items():
    df = pd.read_csv(file_path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)
    display(df.head())

olist_customers_raw (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


olist_orders_raw (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


olist_order_items_raw (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


olist_products_raw (32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


olist_sellers_raw (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


Create typed tables:

In [3]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        cur.execute("""
            DROP TABLE IF EXISTS olist_order_items    CASCADE;
            DROP TABLE IF EXISTS olist_orders         CASCADE;
            DROP TABLE IF EXISTS olist_customers      CASCADE;
            DROP TABLE IF EXISTS olist_products       CASCADE;
            DROP TABLE IF EXISTS olist_sellers        CASCADE;

            CREATE TABLE olist_customers (
                customer_id              TEXT PRIMARY KEY,
                customer_unique_id       TEXT NOT NULL,
                customer_zip_code_prefix INTEGER,
                customer_city            TEXT,
                customer_state           CHAR(2) NOT NULL
            );

            CREATE INDEX idx_olist_customers_unique_id
                ON olist_customers (customer_unique_id);

            CREATE TABLE olist_orders (
                order_id                      TEXT PRIMARY KEY,
                customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
                order_status                  TEXT NOT NULL,
                order_purchase_timestamp      TIMESTAMP NOT NULL,
                order_approved_at             TIMESTAMP,
                order_delivered_carrier_date  TIMESTAMP,
                order_delivered_customer_date TIMESTAMP,
                order_estimated_delivery_date TIMESTAMP,
                CHECK (order_status IN (
                    'created', 'approved', 'invoiced', 'processing',
                    'shipped', 'delivered', 'unavailable', 'canceled'
                ))
            );

            CREATE TABLE olist_products (
                product_id                 TEXT PRIMARY KEY,
                product_category_name      TEXT,
                product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
                product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
                product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
                product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
                product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
                product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
                product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
            );

            CREATE TABLE olist_sellers (
                seller_id              TEXT PRIMARY KEY,
                seller_zip_code_prefix INTEGER,
                seller_city            TEXT,
                seller_state           CHAR(2) NOT NULL
            );

            CREATE TABLE olist_order_items (
                order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
                order_item_id       INTEGER NOT NULL,
                product_id          TEXT REFERENCES olist_products(product_id),
                seller_id           TEXT REFERENCES olist_sellers(seller_id),
                shipping_limit_date TIMESTAMP,
                price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
                freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
                PRIMARY KEY (order_id, order_item_id)
            );

            CREATE INDEX idx_olist_order_items_seller_order
                ON olist_order_items (seller_id, order_id);
        """)
    raw_conn.commit()
except Exception:
    raw_conn.rollback()
    raise
finally:
    raw_conn.close()

### Load typed tables

The raw CSVs are staging data. The typed tables add primary keys, foreign keys, checks, and normalized column types. In the local validation run, all five typed table counts matched their raw counterparts; the count query below makes that comparison visible.


In [4]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_customers
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix::INTEGER,
            customer_city,
            customer_state::CHAR(2)
        FROM olist_customers_raw;

        INSERT INTO olist_orders
        SELECT
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::TIMESTAMP,
            order_approved_at::TIMESTAMP,
            order_delivered_carrier_date::TIMESTAMP,
            order_delivered_customer_date::TIMESTAMP,
            order_estimated_delivery_date::TIMESTAMP
        FROM olist_orders_raw;

        INSERT INTO olist_products
        SELECT
            product_id,
            product_category_name,
            product_name_lenght::INTEGER,
            product_description_lenght::INTEGER,
            product_photos_qty::INTEGER,
            product_weight_g::INTEGER,
            product_length_cm::INTEGER,
            product_height_cm::INTEGER,
            product_width_cm::INTEGER
        FROM olist_products_raw;

        INSERT INTO olist_sellers
        SELECT
            seller_id,
            seller_zip_code_prefix::INTEGER,
            seller_city,
            seller_state::CHAR(2)
        FROM olist_sellers_raw;

        INSERT INTO olist_order_items
        SELECT
            order_id,
            order_item_id::INTEGER,
            product_id,
            seller_id,
            shipping_limit_date::TIMESTAMP,
            price::NUMERIC(12, 2),
            freight_value::NUMERIC(12, 2)
        FROM olist_order_items_raw;
    """))
print('Typed tables loaded; compare their counts with raw tables below.')

Typed tables loaded; compare their counts with raw tables below.


In [5]:
pd.read_sql(text("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
    UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
    UNION ALL SELECT 'products', COUNT(*) FROM olist_products
    UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
    UNION ALL SELECT 'customers_raw' AS table_name, COUNT(*) AS n_rows FROM olist_customers_raw
    UNION ALL SELECT 'orders_raw', COUNT(*) FROM olist_orders_raw
    UNION ALL SELECT 'order_items_raw', COUNT(*) FROM olist_order_items_raw
    UNION ALL SELECT 'products_raw', COUNT(*) FROM olist_products_raw
    UNION ALL SELECT 'sellers_raw', COUNT(*) FROM olist_sellers_raw
    ORDER BY table_name;
"""), engine)

,table_name,n_rows
0,customers,99441
1,customers_raw,99441
2,order_items,112650
3,order_items_raw,112650
4,orders,99441
5,orders_raw,99441
6,products,32951
7,products_raw,32951
8,sellers,3095
9,sellers_raw,3095


## 2. CTE-chain pipeline

Business question: Which previous buyers should receive a category-specific win-back offer? The seven-stage pipeline combines delivered-order recency, frequency, value and recent order size with cancellation share, purchase cohort, and preferred category. Each result row represents one `customer_unique_id` with at least one order. Spending and category features use delivered orders; cancellation rate uses all orders. `avg_order_value_30d` is measured at the latest delivered order, over delivered orders in the preceding 30 days, rather than at the dataset reference date. For buyers without delivered orders, recency, cohort, AOV and preferred category are NULL; frequency and monetary are zero.


In [6]:
with engine.connect() as conn:
    customer_features = pd.read_sql(text("""
WITH
-- Stage 0: set the reference date from the dataset
reference_date AS (
    SELECT MAX(order_purchase_timestamp)::DATE + 1 AS reference_date
    FROM olist_orders
),
-- Stage 1: aggregate item amounts to one row per order
stage1_order_totals AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)
            ::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id, o.customer_id, o.order_id,
             o.order_status, o.order_purchase_timestamp
),
-- Stage 2: summarize each buyer's orders into RFM, cohort and cancellation features
stage2_customer_summary AS (
    SELECT
        s.customer_unique_id,
        MIN(s.customer_id) AS customer_id,
        (r.reference_date - MAX(s.order_date) FILTER (
            WHERE s.order_status = 'delivered'
        ))::INTEGER AS recency_days,
        COUNT(*) FILTER (WHERE s.order_status = 'delivered') AS frequency,
        COALESCE(SUM(s.order_total) FILTER (
            WHERE s.order_status = 'delivered'
        ), 0)::NUMERIC(12, 2) AS monetary,
        DATE_TRUNC('quarter', MIN(s.order_date) FILTER (
            WHERE s.order_status = 'delivered'
        ))::DATE AS cohort_quarter,
        AVG((s.order_status = 'canceled')::INTEGER)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_order_totals AS s
    CROSS JOIN reference_date AS r
    GROUP BY s.customer_unique_id, r.reference_date
),
-- Stage 3: calculate the delivered-order AOV in each rolling 30-day window
stage3_rolling_aov AS (
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_status,
        s.order_ts,
        AVG(s.order_total) FILTER (WHERE s.order_status = 'delivered') OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d
    FROM stage1_order_totals AS s
),
-- Stage 4: keep the AOV at each buyer's latest delivered order
stage4_latest_aov AS (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        avg_order_value_30d
    FROM stage3_rolling_aov
    WHERE order_status = 'delivered'
    ORDER BY customer_unique_id, order_ts DESC, order_id DESC
),
-- Stage 5: aggregate delivered item revenue by buyer and known category
stage5_category_revenue AS (
    SELECT
        s.customer_unique_id,
        p.product_category_name,
        SUM(COALESCE(oi.price, 0))::NUMERIC(12, 2) AS category_revenue
    FROM stage1_order_totals AS s
    JOIN olist_order_items AS oi ON oi.order_id = s.order_id
    JOIN olist_products AS p ON p.product_id = oi.product_id
    WHERE s.order_status = 'delivered'
      AND p.product_category_name IS NOT NULL
    GROUP BY s.customer_unique_id, p.product_category_name
),
-- Stage 6: select each buyer's highest-revenue category
stage6_preferred_category AS (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        product_category_name AS preferred_category
    FROM stage5_category_revenue
    ORDER BY customer_unique_id, category_revenue DESC, product_category_name
)
SELECT
    s.customer_unique_id,
    s.recency_days,
    s.frequency,
    s.monetary,
    s.cohort_quarter,
    a.avg_order_value_30d,
    s.cancellation_rate,
    p.preferred_category
FROM stage2_customer_summary AS s
LEFT JOIN stage4_latest_aov AS a USING (customer_unique_id)
LEFT JOIN stage6_preferred_category AS p USING (customer_unique_id)
ORDER BY s.customer_unique_id;
    """), conn)

print(f"Customer rows: {len(customer_features):,}; unique buyers: "
      f"{customer_features['customer_unique_id'].nunique():,}")
display(customer_features.head(20))


Customer rows: 96,096; unique buyers: 96,096


,customer_unique_id,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d,cancellation_rate,preferred_category
0,0000366f3b9a7992bf8c76cfdf3221e2,161.0,1,141.90,2018-04-01,141.90,0.0,cama_mesa_banho
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164.0,1,27.19,2018-04-01,27.19,0.0,beleza_saude
2,0000f46a3911fa3c0805444483337064,587.0,1,86.22,2017-01-01,86.22,0.0,papelaria
3,0000f6ccb0745a6a4b88665a16c9f078,371.0,1,43.62,2017-10-01,43.62,0.0,telefonia
4,0004aac84e0df4da2b147fca70cf8255,338.0,1,196.89,2017-10-01,196.89,0.0,telefonia
5,0004bd2a26a76fe21f786e4fbd80607f,196.0,1,166.98,2018-04-01,166.98,0.0,ferramentas_jardim
6,00050ab1314c0e55a6ca13cf7181fecf,181.0,1,35.38,2018-04-01,35.38,0.0,telefonia
7,00053a61a98854899e70ed204dd4bafe,232.0,1,419.18,2018-01-01,419.18,0.0,esporte_lazer
8,0005e1862207bf6ccc02e4228effd9a0,593.0,1,150.12,2017-01-01,150.12,0.0,fashion_bolsas_e_acessorios
9,0005ef4cd20d2893f0d9fbd94d3c0d97,220.0,1,129.76,2018-01-01,129.76,0.0,esporte_lazer


## 3. Recursive CTE for referral chain

In [7]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
    DROP TABLE IF EXISTS hw5_referrals CASCADE;

    CREATE TABLE hw5_referrals (
        referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
        referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
        referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        PRIMARY KEY (referred_customer_id),
        CHECK (referrer_customer_id <> referred_customer_id)
    );

    WITH ordered_customers AS (
        SELECT
            customer_id,
            ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
        FROM olist_customers
        ORDER BY customer_id
        LIMIT 10
    )
    INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
    SELECT
        prev.customer_id AS referrer_customer_id,
        nxt.customer_id AS referred_customer_id
    FROM ordered_customers AS prev
    JOIN ordered_customers AS nxt
        ON nxt.rn = prev.rn + 1
    ON CONFLICT (referred_customer_id) DO NOTHING;
    """)

display(pd.read_sql(text("""
    SELECT *
    FROM hw5_referrals
    ORDER BY referrer_customer_id, referred_customer_id;
"""), engine))


,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-24 10:02:28.115714+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-24 10:02:28.115714+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-24 10:02:28.115714+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-24 10:02:28.115714+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-24 10:02:28.115714+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-24 10:02:28.115714+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-24 10:02:28.115714+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-24 10:02:28.115714+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-24 10:02:28.115714+00:00


In [8]:
pd.read_sql(text("""
    WITH RECURSIVE referral_depth AS (
        -- ANCHOR: root customers, які не були referred іншими customers
        SELECT
            c.customer_id,
            1 AS depth,
            ARRAY[c.customer_id]::TEXT[] AS path_ids,
            c.customer_id::TEXT AS path_text
        FROM olist_customers AS c
        WHERE NOT EXISTS (
            SELECT 1
            FROM hw5_referrals AS r
            WHERE r.referred_customer_id = c.customer_id
        )

        UNION ALL

        -- RECURSIVE: перейти від referrer до його referred customers
        SELECT
            r.referred_customer_id AS customer_id,
            rd.depth + 1 AS depth,
            rd.path_ids || r.referred_customer_id AS path_ids,
            rd.path_text || ' -> ' || r.referred_customer_id AS path_text
        FROM hw5_referrals AS r
        JOIN referral_depth AS rd
            ON rd.customer_id = r.referrer_customer_id
        WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
        AND rd.depth < 50
    )
    SELECT
        depth,
        COUNT(*) AS n_customers,
        MAX(path_text) AS sample_path
    FROM referral_depth
    GROUP BY depth
    ORDER BY depth;
"""), engine)


,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


## 4. Materialized View with CONCURRENT REFRESH


The view reproduces all Section 2 features at the `customer_unique_id` level. Olist's `customer_id` identifies an order-specific customer record, so the view uses `MIN(customer_id)` as a deterministic representative for each buyer and keeps the required unique index on that column. Other `customer_id` values for the same buyer do not appear in the view. For a lookup by any Olist `customer_id`, first use `olist_customers` to find its `customer_unique_id`, then join to the view on that buyer key. A second unique index supports this lookup. The representative ID can change if new customer records are added.


In [9]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
        DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
        CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
-- Stage 0: set the reference date from the dataset
reference_date AS (
    SELECT MAX(order_purchase_timestamp)::DATE + 1 AS reference_date
    FROM olist_orders
),
-- Stage 1: aggregate item amounts to one row per order
stage1_order_totals AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)
            ::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id, o.customer_id, o.order_id,
             o.order_status, o.order_purchase_timestamp
),
-- Stage 2: summarize each buyer's orders into RFM, cohort and cancellation features
stage2_customer_summary AS (
    SELECT
        s.customer_unique_id,
        MIN(s.customer_id) AS customer_id,
        (r.reference_date - MAX(s.order_date) FILTER (
            WHERE s.order_status = 'delivered'
        ))::INTEGER AS recency_days,
        COUNT(*) FILTER (WHERE s.order_status = 'delivered') AS frequency,
        COALESCE(SUM(s.order_total) FILTER (
            WHERE s.order_status = 'delivered'
        ), 0)::NUMERIC(12, 2) AS monetary,
        DATE_TRUNC('quarter', MIN(s.order_date) FILTER (
            WHERE s.order_status = 'delivered'
        ))::DATE AS cohort_quarter,
        AVG((s.order_status = 'canceled')::INTEGER)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_order_totals AS s
    CROSS JOIN reference_date AS r
    GROUP BY s.customer_unique_id, r.reference_date
),
-- Stage 3: calculate the delivered-order AOV in each rolling 30-day window
stage3_rolling_aov AS (
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_status,
        s.order_ts,
        AVG(s.order_total) FILTER (WHERE s.order_status = 'delivered') OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d
    FROM stage1_order_totals AS s
),
-- Stage 4: keep the AOV at each buyer's latest delivered order
stage4_latest_aov AS (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        avg_order_value_30d
    FROM stage3_rolling_aov
    WHERE order_status = 'delivered'
    ORDER BY customer_unique_id, order_ts DESC, order_id DESC
),
-- Stage 5: aggregate delivered item revenue by buyer and known category
stage5_category_revenue AS (
    SELECT
        s.customer_unique_id,
        p.product_category_name,
        SUM(COALESCE(oi.price, 0))::NUMERIC(12, 2) AS category_revenue
    FROM stage1_order_totals AS s
    JOIN olist_order_items AS oi ON oi.order_id = s.order_id
    JOIN olist_products AS p ON p.product_id = oi.product_id
    WHERE s.order_status = 'delivered'
      AND p.product_category_name IS NOT NULL
    GROUP BY s.customer_unique_id, p.product_category_name
),
-- Stage 6: select each buyer's highest-revenue category
stage6_preferred_category AS (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        product_category_name AS preferred_category
    FROM stage5_category_revenue
    ORDER BY customer_unique_id, category_revenue DESC, product_category_name
)
SELECT
    s.customer_id,
    s.customer_unique_id,
    s.recency_days,
    s.frequency,
    s.monetary,
    s.cohort_quarter,
    a.avg_order_value_30d,
    s.cancellation_rate,
    p.preferred_category
FROM stage2_customer_summary AS s
LEFT JOIN stage4_latest_aov AS a USING (customer_unique_id)
LEFT JOIN stage6_preferred_category AS p USING (customer_unique_id);
        CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
            ON mv_olist_customer_features (customer_id);
        CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_unique_id
            ON mv_olist_customer_features (customer_unique_id);
    """)

print("Created mv_olist_customer_features and both unique indexes.")


Created mv_olist_customer_features and both unique indexes.


### Validate the materialized view


In [10]:
with engine.connect() as conn:
    mv_counts = pd.read_sql(text("""
        SELECT
            COUNT(*) AS n_rows,
            COUNT(DISTINCT customer_id) AS distinct_customer_ids,
            COUNT(DISTINCT customer_unique_id) AS distinct_buyers
        FROM mv_olist_customer_features
    """), conn)
    mv_top5 = pd.read_sql(text("""
        SELECT *
        FROM mv_olist_customer_features
        ORDER BY monetary DESC NULLS LAST, customer_id
        LIMIT 5
    """), conn)

display(mv_counts)
display(mv_top5)


,n_rows,distinct_customer_ids,distinct_buyers
0,96096,96096,96096


,customer_id,customer_unique_id,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d,cancellation_rate,preferred_category
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,384,1,13664.08,2017-07-01,13664.08,0.0,telefonia_fixa
1,349509b216bd5ec11c5fae929fd13595,da122df9eeddfedc1dc1f5349a1a690c,565,2,7571.63,2017-04-01,3785.82,0.0,eletroportateis
2,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,95,1,7274.88,2018-07-01,7274.88,0.0,telefonia_fixa
3,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,613,1,6929.31,2017-01-01,6929.31,0.0,utilidades_domesticas
4,f48d464a0baaea338cb25f816991ab1f,459bef486812aa25204be022145caa62,85,1,6922.21,2018-07-01,6922.21,0.0,pcs


### Lookup by any Olist `customer_id`
The example chooses a nonrepresentative ID, then resolves it through `customer_unique_id`. The coverage check confirms that every source `customer_id` resolves to one buyer row.


In [11]:
# Pick an order-specific ID that is not the buyer's representative ID.
with engine.connect() as conn:
    target_customer_id = conn.execute(text("""
        SELECT MAX(customer_id)
        FROM olist_customers
        GROUP BY customer_unique_id
        HAVING COUNT(*) > 1
        ORDER BY customer_unique_id
        LIMIT 1
    """)).scalar_one()

    buyer_lookup = pd.read_sql(text("""
        SELECT
            c.customer_id AS requested_customer_id,
            mv.customer_id AS representative_customer_id,
            mv.customer_unique_id,
            mv.recency_days,
            mv.frequency,
            mv.monetary,
            mv.cohort_quarter,
            mv.avg_order_value_30d,
            mv.cancellation_rate,
            mv.preferred_category
        FROM olist_customers AS c
        JOIN mv_olist_customer_features AS mv
            ON mv.customer_unique_id = c.customer_unique_id
        WHERE c.customer_id = :customer_id
    """), conn, params={"customer_id": target_customer_id})

    lookup_coverage = pd.read_sql(text("""
        SELECT
            COUNT(*) AS customer_ids,
            COUNT(mv.customer_unique_id) AS resolved_ids,
            COUNT(*) FILTER (WHERE c.customer_id <> mv.customer_id)
                AS nonrepresentative_ids
        FROM olist_customers AS c
        LEFT JOIN mv_olist_customer_features AS mv
            ON mv.customer_unique_id = c.customer_unique_id
    """), conn)

assert len(buyer_lookup) == 1
assert buyer_lookup.loc[0, 'requested_customer_id'] != buyer_lookup.loc[0, 'representative_customer_id']
assert lookup_coverage.loc[0, 'customer_ids'] == lookup_coverage.loc[0, 'resolved_ids']
print("Buyer features for a nonrepresentative customer_id:")
display(buyer_lookup.T)
print("Lookup coverage across all order-specific customer IDs:")
display(lookup_coverage)


Buyer features for a nonrepresentative customer_id:


,0
requested_customer_id,24b0e2bd287e47d54d193e7bbb51103f
representative_customer_id,1afe8a9c67eec3516c09a8bdcc539090
customer_unique_id,00172711b30d52eea8b313a7f2cced02
recency_days,82
frequency,1
monetary,122.07
cohort_quarter,2018-07-01
avg_order_value_30d,122.07
cancellation_rate,0.5
preferred_category,papelaria


Lookup coverage across all order-specific customer IDs:


,customer_ids,resolved_ids,nonrepresentative_ids
0,99441,99441,3345


### Refresh concurrently
Run this cell after the unique index has been created.


In [12]:
with engine.begin() as conn:
    conn.exec_driver_sql(
        "REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features"
    )

print("Concurrent refresh completed.")


Concurrent refresh completed.


## 5. EXPLAIN ANALYZE

The plans below use fresh PostgreSQL statistics for the typed tables and materialized view. Execution times depend on the machine and cache state.


### Update planner statistics
Run after loading the typed tables and refreshing the materialized view.


In [13]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
        ANALYZE olist_customers;
        ANALYZE olist_orders;
        ANALYZE olist_order_items;
        ANALYZE olist_products;
        ANALYZE olist_sellers;
        ANALYZE mv_olist_customer_features;
    """)

print("Updated planner statistics for the typed tables and materialized view.")


Updated planner statistics for the typed tables and materialized view.


### Full Section 2 pipeline: top 10 by monetary value


In [14]:
with engine.connect() as conn:
    plan = conn.execute(text("""
EXPLAIN (ANALYZE, BUFFERS)
WITH
-- Stage 0: set the reference date from the dataset
reference_date AS (
    SELECT MAX(order_purchase_timestamp)::DATE + 1 AS reference_date
    FROM olist_orders
),
-- Stage 1: aggregate item amounts to one row per order
stage1_order_totals AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)
            ::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id, o.customer_id, o.order_id,
             o.order_status, o.order_purchase_timestamp
),
-- Stage 2: summarize each buyer's orders into RFM, cohort and cancellation features
stage2_customer_summary AS (
    SELECT
        s.customer_unique_id,
        MIN(s.customer_id) AS customer_id,
        (r.reference_date - MAX(s.order_date) FILTER (
            WHERE s.order_status = 'delivered'
        ))::INTEGER AS recency_days,
        COUNT(*) FILTER (WHERE s.order_status = 'delivered') AS frequency,
        COALESCE(SUM(s.order_total) FILTER (
            WHERE s.order_status = 'delivered'
        ), 0)::NUMERIC(12, 2) AS monetary,
        DATE_TRUNC('quarter', MIN(s.order_date) FILTER (
            WHERE s.order_status = 'delivered'
        ))::DATE AS cohort_quarter,
        AVG((s.order_status = 'canceled')::INTEGER)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_order_totals AS s
    CROSS JOIN reference_date AS r
    GROUP BY s.customer_unique_id, r.reference_date
),
-- Stage 3: calculate the delivered-order AOV in each rolling 30-day window
stage3_rolling_aov AS (
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_status,
        s.order_ts,
        AVG(s.order_total) FILTER (WHERE s.order_status = 'delivered') OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d
    FROM stage1_order_totals AS s
),
-- Stage 4: keep the AOV at each buyer's latest delivered order
stage4_latest_aov AS (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        avg_order_value_30d
    FROM stage3_rolling_aov
    WHERE order_status = 'delivered'
    ORDER BY customer_unique_id, order_ts DESC, order_id DESC
),
-- Stage 5: aggregate delivered item revenue by buyer and known category
stage5_category_revenue AS (
    SELECT
        s.customer_unique_id,
        p.product_category_name,
        SUM(COALESCE(oi.price, 0))::NUMERIC(12, 2) AS category_revenue
    FROM stage1_order_totals AS s
    JOIN olist_order_items AS oi ON oi.order_id = s.order_id
    JOIN olist_products AS p ON p.product_id = oi.product_id
    WHERE s.order_status = 'delivered'
      AND p.product_category_name IS NOT NULL
    GROUP BY s.customer_unique_id, p.product_category_name
),
-- Stage 6: select each buyer's highest-revenue category
stage6_preferred_category AS (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        product_category_name AS preferred_category
    FROM stage5_category_revenue
    ORDER BY customer_unique_id, category_revenue DESC, product_category_name
)
SELECT
    s.customer_unique_id,
    s.recency_days,
    s.frequency,
    s.monetary,
    s.cohort_quarter,
    a.avg_order_value_30d,
    s.cancellation_rate,
    p.preferred_category
FROM stage2_customer_summary AS s
LEFT JOIN stage4_latest_aov AS a USING (customer_unique_id)
LEFT JOIN stage6_preferred_category AS p USING (customer_unique_id)
ORDER BY monetary DESC NULLS LAST, s.customer_unique_id
LIMIT 10;
    """)).scalars().all()

print("\n".join(plan))


Limit  (cost=71430.96..71430.98 rows=10 width=109) (actual time=1270.158..1270.665 rows=10 loops=1)
  Buffers: shared hit=834033 read=335, temp read=9643 written=8678
  CTE stage1_order_totals
    ->  Finalize GroupAggregate  (cost=19811.33..31556.79 rows=112650 width=136) (actual time=106.068..202.036 rows=99441 loops=1)
          Group Key: c.customer_unique_id, o.order_id
          Buffers: shared hit=5608, temp read=1990 written=1994
          ->  Gather Merge  (cost=19811.33..28922.76 rows=66265 width=148) (actual time=106.062..156.346 rows=99674 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=5608, temp read=1990 written=1994
                ->  Partial GroupAggregate  (cost=18811.32..20467.94 rows=66265 width=148) (actual time=101.774..136.300 rows=49837 loops=2)
                      Group Key: c.customer_unique_id, o.order_id
                      Buffers: shared hit=5608, temp read=1990 written=1994
         

### Materialized view: representative `customer_id` lookup
This plan tests the assignment-required unique index on the MV. For an arbitrary Olist `customer_id`, use the buyer-key lookup in Section 4.


In [15]:
with engine.connect() as conn:
    plan = conn.execute(text("""
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t ON t.customer_id = mv.customer_id;
    """)).scalars().all()

print("\n".join(plan))


Nested Loop  (cost=0.83..8.96 rows=1 width=112) (actual time=0.011..0.011 rows=1 loops=1)
  Buffers: shared hit=8
  ->  Limit  (cost=0.42..0.53 rows=1 width=33) (actual time=0.006..0.007 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..11127.92 rows=99441 width=33) (actual time=0.006..0.006 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=112) (actual time=0.003..0.003 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=4
Planning:
  Buffers: shared hit=40 read=1
Planning Time: 0.197 ms
Execution Time: 0.022 ms


### Reflection

The costliest feature is `preferred_category`. Its revenue and top-category branch reaches about 537 ms in the saved plan. Nested-loop joins connect 96,478 delivered orders to items and products; the two primary-key `Index Scan` nodes run 96,478 and 110,197 times. This branch records about 826,600 shared buffer hits. Its external-merge `Sort` spills roughly 6.9 MB to disk before `GroupAggregate` calculates category revenue, and an incremental sort selects the top category. By comparison, the buyer-summary `HashAggregate`, which computes RFM, cohort and cancellation rate together, reaches about 391 ms and spills across 21 batches. The rolling AOV selection reaches about 192 ms. The final top-10 sort uses only 27 kB.

Materializing this customer-level result is useful for repeated campaign lookups. The full seven-stage query took 1,275 ms; the representative-`customer_id` MV lookup took 0.022 ms and used its unique index. The queries return different row counts, so these times are not a direct speedup ratio. For active commerce data, I would refresh daily after ingestion: a win-back campaign benefits from recent orders, while hourly refresh would repeatedly pay the join and sort costs. A static historical archive could refresh on demand.

A regular view is preferable for a support screen that must show a cancellation immediately after an order status change. It reads current tables and accepts recomputation latency.

The referral CTE rejects any `next_id` already present in `path_ids` and caps traversal with `rd.depth < 50`. An A → B → A cycle has no root under the current anchor and is omitted. If seeded inside that cycle, the path guard rejects the second A; the depth guard alone would stop after 50 levels. Without both, `UNION ALL` could recurse indefinitely.

The main trade-off is lookup latency versus freshness. Daily refresh bounds staleness to about one day; storage cost is secondary.
